# 多GPU训练实战

本notebook深入讲解多GPU训练的实现细节和分布式训练策略。

## 学习目标

- 从零实现多GPU数据并行
- 掌握PyTorch高级多GPU API
- 理解参数服务器架构
- 学习分布式训练最佳实践

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import time
import numpy as np

print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

## 1. 从零实现数据并行

### 1.1 核心组件

实现数据并行需要4个关键函数:

1. **get_params**: 复制参数到各GPU
2. **split_batch**: 分割数据到各GPU
3. **allreduce**: 聚合梯度
4. **train_batch**: 训练一个批次

### 1.2 参数分发

In [ ]:
def get_params(params, device):
    """
    将参数复制到指定设备并附加梯度
    
    参数:
        params: 参数列表
        device: 目标设备
    
    返回:
        new_params: 在新设备上的参数(带梯度)
    """
    new_params = [p.to(device) for p in params]
    for p in new_params:
        p.requires_grad_()
    return new_params

# 演示
if torch.cuda.is_available():
    # 创建一些参数
    params = [torch.randn(10, 10) for _ in range(3)]
    
    # 复制到GPU 0
    device = torch.device('cuda:0')
    new_params = get_params(params, device)
    
    print(f"原参数设备: {params[0].device}")
    print(f"新参数设备: {new_params[0].device}")
    print(f"需要梯度: {new_params[0].requires_grad}")
else:
    print("需要GPU来演示")

### 1.3 梯度聚合 (All-Reduce)

In [ ]:
def allreduce(data):
    """
    All-Reduce操作: 将所有设备上的数据求和并广播
    
    参数:
        data: 列表,每个元素是一个设备上的tensor
    
    操作:
        1. 将所有tensor求和到data[0]的设备
        2. 将结果广播到所有设备
    """
    # 步骤1: 累加到第一个设备
    for i in range(1, len(data)):
        data[0][:] += data[i].to(data[0].device)
    
    # 步骤2: 广播到所有设备
    for i in range(1, len(data)):
        data[i][:] = data[0].to(data[i].device)

# 演示All-Reduce
if torch.cuda.device_count() >= 2:
    # 在不同GPU上创建不同值
    data = [torch.ones((2, 3), device=f'cuda:{i}') * (i + 1) 
            for i in range(2)]
    
    print("All-Reduce之前:")
    print(f"  GPU 0: {data[0]}")
    print(f"  GPU 1: {data[1]}")
    
    allreduce(data)
    
    print("\nAll-Reduce之后:")
    print(f"  GPU 0: {data[0]}")
    print(f"  GPU 1: {data[1]}")
    print("\n注意: 所有GPU现在都有相同的值(原值之和)")
elif torch.cuda.is_available():
    print("需要至少2个GPU来演示All-Reduce")
else:
    print("需要GPU")

### 1.4 数据分割

In [ ]:
def split_batch(X, y, devices):
    """
    将批次数据均匀分割到多个设备
    
    参数:
        X: 输入数据 [batch_size, ...]
        y: 标签 [batch_size]
        devices: 设备列表
    
    返回:
        X_shards: 分割后的输入列表
        y_shards: 分割后的标签列表
    """
    assert X.shape[0] == y.shape[0]
    return (nn.parallel.scatter(X, devices),
            nn.parallel.scatter(y, devices))

# 演示数据分割
if torch.cuda.device_count() >= 2:
    # 创建示例数据
    X = torch.arange(20).reshape(4, 5)  # 4个样本
    y = torch.arange(4)
    
    devices = [torch.device(f'cuda:{i}') for i in range(2)]
    X_shards, y_shards = split_batch(X, y, devices)
    
    print(f"原始数据形状: X={X.shape}, y={y.shape}")
    print(f"\n分割后(2个GPU):")
    for i, (x_shard, y_shard) in enumerate(zip(X_shards, y_shards)):
        print(f"  GPU {i}: X形状={x_shard.shape}, y={y_shard}")
elif torch.cuda.is_available():
    print("需要至少2个GPU")
else:
    print("需要GPU")

### 1.5 完整训练步骤

整合所有组件,实现一个批次的训练:

In [ ]:
def train_batch(X, y, device_params, devices, lr, net, loss_fn):
    """
    在多个GPU上训练一个批次
    
    参数:
        X, y: 批次数据
        device_params: 每个设备上的参数列表
        devices: 设备列表
        lr: 学习率
        net: 网络函数
        loss_fn: 损失函数
    
    步骤:
        1. 分割数据到各GPU
        2. 各GPU并行前向传播
        3. 各GPU并行反向传播
        4. All-Reduce聚合梯度
        5. 各GPU更新参数
    """
    # 步骤1: 分割数据
    X_shards, y_shards = split_batch(X, y, devices)
    
    # 步骤2-3: 并行前向和反向传播
    losses = []
    for X_shard, y_shard, params in zip(X_shards, y_shards, device_params):
        # 前向传播
        y_hat = net(X_shard, params)
        l = loss_fn(y_hat, y_shard).sum()
        
        # 反向传播
        l.backward()
        losses.append(l)
    
    # 步骤4: All-Reduce聚合梯度
    with torch.no_grad():
        for i in range(len(device_params[0])):
            allreduce([device_params[c][i].grad 
                      for c in range(len(devices))])
    
    # 步骤5: 更新参数(SGD)
    with torch.no_grad():
        for params in device_params:
            for param in params:
                param[:] -= lr * param.grad / X.shape[0]
                param.grad.zero_()
    
    return sum(l.item() for l in losses) / len(losses)

print("train_batch函数完成!")
print("\n关键步骤:")
print("1. split_batch: 分割数据")
print("2. 各GPU并行前向传播")
print("3. 各GPU并行反向传播")
print("4. allreduce: 聚合梯度")
print("5. 各GPU更新参数")

## 2. 实战: 多GPU训练LeNet

### 2.1 定义网络

In [ ]:
# 初始化LeNet参数
def init_lenet_params(scale=0.01):
    """初始化LeNet参数"""
    W1 = torch.randn(20, 1, 3, 3) * scale  # Conv1
    b1 = torch.zeros(20)
    W2 = torch.randn(50, 20, 5, 5) * scale  # Conv2
    b2 = torch.zeros(50)
    W3 = torch.randn(800, 128) * scale  # FC1
    b3 = torch.zeros(128)
    W4 = torch.randn(128, 10) * scale  # FC2
    b4 = torch.zeros(10)
    return [W1, b1, W2, b2, W3, b3, W4, b4]

def lenet(X, params):
    """
    LeNet前向传播
    
    架构:
        Conv(20,3x3) -> ReLU -> AvgPool(2x2) ->
        Conv(50,5x5) -> ReLU -> AvgPool(2x2) ->
        Flatten -> FC(128) -> ReLU -> FC(10)
    """
    # 第一层卷积
    h1 = F.conv2d(X, params[0], params[1])
    h1 = F.relu(h1)
    h1 = F.avg_pool2d(h1, kernel_size=2, stride=2)
    
    # 第二层卷积
    h2 = F.conv2d(h1, params[2], params[3])
    h2 = F.relu(h2)
    h2 = F.avg_pool2d(h2, kernel_size=2, stride=2)
    
    # 展平
    h2 = h2.reshape(h2.shape[0], -1)
    
    # 全连接层
    h3 = torch.mm(h2, params[4]) + params[5]
    h3 = F.relu(h3)
    
    # 输出层
    y_hat = torch.mm(h3, params[6]) + params[7]
    return y_hat

print("LeNet定义完成!")
print("\n参数数量:")
params = init_lenet_params()
total_params = sum(p.numel() for p in params)
print(f"  总计: {total_params:,} 参数")

### 2.2 准备数据

In [ ]:
def load_data_fashion_mnist(batch_size, resize=None):
    """下载Fashion-MNIST数据集"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    
    mnist_train = torchvision.datasets.FashionMNIST(
        root='./data', train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root='./data', train=False, transform=trans, download=True)
    
    train_iter = DataLoader(mnist_train, batch_size, 
                           shuffle=True, num_workers=4)
    test_iter = DataLoader(mnist_test, batch_size, 
                          shuffle=False, num_workers=4)
    
    return train_iter, test_iter

print("数据加载函数定义完成")

### 2.3 训练函数

In [ ]:
def evaluate_accuracy(net, data_iter, params, device):
    """计算模型精度"""
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in data_iter:
            X, y = X.to(device), y.to(device)
            y_hat = net(X, params)
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return correct / total

def train_multi_gpu(num_gpus, batch_size, lr, num_epochs=10):
    """
    多GPU训练主函数
    
    参数:
        num_gpus: GPU数量
        batch_size: 批大小(总批大小,会分配到各GPU)
        lr: 学习率
        num_epochs: 训练轮数
    """
    # 准备数据
    train_iter, test_iter = load_data_fashion_mnist(batch_size)
    
    # 准备设备
    if torch.cuda.is_available() and num_gpus > 0:
        devices = [torch.device(f'cuda:{i}') 
                  for i in range(min(num_gpus, torch.cuda.device_count()))]
    else:
        devices = [torch.device('cpu')]
        num_gpus = 1
    
    print(f"使用设备: {devices}")
    
    # 初始化参数并复制到各GPU
    params = init_lenet_params()
    device_params = [get_params(params, d) for d in devices]
    
    # 损失函数
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    
    # 训练
    print(f"\n开始训练 ({num_epochs} epochs)...")
    for epoch in range(num_epochs):
        start_time = time.time()
        train_loss = 0
        
        for X, y in train_iter:
            loss = train_batch(X, y, device_params, devices, 
                             lr, lenet, loss_fn)
            train_loss += loss
            
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        
        # 评估
        test_acc = evaluate_accuracy(lenet, test_iter, 
                                    device_params[0], devices[0])
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Loss={train_loss/len(train_iter):.4f}, "
              f"Test Acc={test_acc:.4f}, "
              f"Time={epoch_time:.2f}s")
    
    print(f"\n训练完成! 最终测试精度: {test_acc:.4f}")
    return test_acc

print("训练函数定义完成!")

### 2.4 运行训练

**注意**: 由于从零实现开销较大,这里用小数据集快速演示。生产环境请使用PyTorch内置API。

In [ ]:
# 单GPU训练
if torch.cuda.is_available():
    print("=== 单GPU训练 ===")
    acc_1gpu = train_multi_gpu(num_gpus=1, batch_size=256, lr=0.2, num_epochs=3)
else:
    print("没有GPU,跳过训练演示")

In [ ]:
# 多GPU训练
if torch.cuda.device_count() >= 2:
    print("\n=== 2-GPU训练 ===")
    acc_2gpu = train_multi_gpu(num_gpus=2, batch_size=256, lr=0.2, num_epochs=3)
    
    print(f"\n精度对比:")
    print(f"  单GPU: {acc_1gpu:.4f}")
    print(f"  2-GPU: {acc_2gpu:.4f}")
    print("\n注意: 精度应该相近,因为算法本质相同")
elif torch.cuda.is_available():
    print("\n只有1个GPU,无法演示多GPU训练")
else:
    print("没有GPU")

## 3. PyTorch高级API

### 3.1 nn.DataParallel (简洁实现)

In [ ]:
# 定义模型(使用nn.Module)
class LeNetModule(nn.Module):
    """LeNet-5网络"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 20, 3, padding=1)
        self.conv2 = nn.Conv2d(20, 50, 5)
        self.fc1 = nn.Linear(800, 128)
        self.fc2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = F.avg_pool2d(F.relu(self.conv1(x)), 2)
        x = F.avg_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# 使用DataParallel
def train_with_dataparallel(batch_size=256, lr=0.1, num_epochs=3):
    """使用nn.DataParallel训练"""
    # 准备数据
    train_iter, test_iter = load_data_fashion_mnist(batch_size)
    
    # 创建模型
    model = LeNetModule()
    
    # 多GPU包装
    if torch.cuda.device_count() > 1:
        print(f"使用 {torch.cuda.device_count()} 个GPU (DataParallel)")
        model = nn.DataParallel(model)
    
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    # 优化器和损失
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # 训练
    print("\n开始训练...")
    for epoch in range(num_epochs):
        model.train()
        start_time = time.time()
        train_loss = 0
        
        for X, y in train_iter:
            X, y = X.to(device), y.to(device)
            
            optimizer.zero_grad()
            output = model(X)  # DataParallel自动分割数据!
            loss = criterion(output, y)
            loss.backward()  # DataParallel自动聚合梯度!
            optimizer.step()
            
            train_loss += loss.item()
        
        # 评估
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for X, y in test_iter:
                X, y = X.to(device), y.to(device)
                output = model(X)
                correct += (output.argmax(1) == y).sum().item()
                total += y.size(0)
        
        test_acc = correct / total
        epoch_time = time.time() - start_time
        
        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Loss={train_loss/len(train_iter):.4f}, "
              f"Test Acc={test_acc:.4f}, "
              f"Time={epoch_time:.2f}s")
    
    print(f"\nDataParallel训练完成! 测试精度: {test_acc:.4f}")
    return test_acc

# 运行
if torch.cuda.is_available():
    print("=== 使用nn.DataParallel ===")
    acc_dp = train_with_dataparallel()
else:
    print("需要GPU")

### 3.2 ResNet-18 多GPU训练示例

使用更大的模型展示多GPU优势:

In [ ]:
# ResNet基本块
class Residual(nn.Module):
    """ResNet残差块"""
    def __init__(self, in_channels, out_channels, use_1x1conv=False, strides=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 
                              kernel_size=3, padding=1, stride=strides)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 
                              kernel_size=3, padding=1)
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, 
                                  kernel_size=1, stride=strides)
        else:
            self.conv3 = None
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
    
    def forward(self, X):
        Y = F.relu(self.bn1(self.conv1(X)))
        Y = self.bn2(self.conv2(Y))
        if self.conv3:
            X = self.conv3(X)
        Y += X
        return F.relu(Y)

def resnet18(num_classes=10):
    """简化的ResNet-18"""
    def resnet_block(in_channels, out_channels, num_residuals, first_block=False):
        blk = []
        for i in range(num_residuals):
            if i == 0 and not first_block:
                blk.append(Residual(in_channels, out_channels, 
                                   use_1x1conv=True, strides=2))
            else:
                blk.append(Residual(out_channels, out_channels))
        return nn.Sequential(*blk)
    
    net = nn.Sequential(
        nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(64),
        nn.ReLU(),
        resnet_block(64, 64, 2, first_block=True),
        resnet_block(64, 128, 2),
        resnet_block(128, 256, 2),
        resnet_block(256, 512, 2),
        nn.AdaptiveAvgPool2d((1, 1)),
        nn.Flatten(),
        nn.Linear(512, num_classes)
    )
    return net

print("ResNet-18定义完成!")
print("\n参数量:")
model = resnet18()
total_params = sum(p.numel() for p in model.parameters())
print(f"  总计: {total_params:,} 参数")
print(f"  约 {total_params/1e6:.2f}M 参数")

## 4. 参数服务器架构

### 4.1 参数服务器概念

**传统方法**: All-Reduce (所有GPU直接通信)

**参数服务器**: 中心化的参数管理

```
┌─────────┐
│Parameter│  ← 存储参数
│ Server  │  ← 聚合梯度
└────┬────┘  ← 分发更新
     │
  ┌──┴──┐
  │     │
Worker Worker  ← 计算梯度
GPU 0  GPU 1   ← 前向/反向传播
```

### 4.2 工作流程

**Push-Pull模型**:

1. **Pull**: Worker从服务器拉取参数
   ```python
   params = server.pull(keys)
   ```

2. **Compute**: Worker计算梯度
   ```python
   grads = compute_gradients(params, data)
   ```

3. **Push**: Worker推送梯度到服务器
   ```python
   server.push(keys, grads)
   ```

4. **Aggregate**: 服务器聚合梯度并更新
   ```python
   aggregated_grads = sum(all_grads) / num_workers
   params -= lr * aggregated_grads
   ```

### 4.3 多参数服务器

**单服务器瓶颈**: 带宽限制 $O(m)$ (m=worker数)

**解决方案**: 参数分片到多个服务器

```
Server 0: 参数 [0:n/2]
Server 1: 参数 [n/2:n]

每个worker:
  - 向Server 0推送/拉取前半参数
  - 向Server 1推送/拉取后半参数
  - 并行通信!
```

**带宽**: $O(m/n)$ (n=服务器数)

### 4.4 键值存储抽象

**接口**:
```python
class KVStore:
    def push(key, value):
        """累积value到key对应的存储"""
        storage[key] += value
    
    def pull(key):
        """获取key对应的聚合值"""
        return storage[key]
```

**优势**:
- 简单API隐藏复杂性
- 支持异步操作
- 可扩展到多机

### 4.5 All-Reduce vs 参数服务器

| 特性 | All-Reduce | 参数服务器 |
|------|-----------|------------|
| 通信复杂度 | O(1) | O(m/n) |
| 单点故障 | 无 | 有 |
| 实现复杂度 | 低 | 中 |
| 扩展性 | 优秀 | 良好 |
| 异步支持 | 困难 | 容易 |
| 推荐场景 | 单机多GPU | 多机训练 |

**结论**: 
- **单机多GPU**: All-Reduce (NCCL)
- **多机训练**: 可考虑参数服务器或All-Reduce
- **超大规模**: 混合方案(机内All-Reduce + 机间PS)

## 5. 分布式训练最佳实践

### 5.1 性能优化策略

#### 1. 梯度累积 (Gradient Accumulation)

**问题**: 显存不足,无法用大批量

**解决**: 累积多个小批次的梯度

```python
accumulation_steps = 4

for i, (X, y) in enumerate(dataloader):
    output = model(X)
    loss = criterion(output, y) / accumulation_steps
    loss.backward()  # 累积梯度
    
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()  # 更新参数
        optimizer.zero_grad()  # 清空梯度
```

**效果**: 等价批量 = batch_size × accumulation_steps

#### 2. 混合精度训练 (Mixed Precision)

**优势**:
- 减少显存占用(FP16 vs FP32)
- 加速计算(Tensor Core优化)
- 减少通信量(梯度更小)

```python
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for X, y in dataloader:
    with autocast():  # 自动FP16
        output = model(X)
        loss = criterion(output, y)
    
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
```

#### 3. 梯度压缩

**技术**:
- **量化**: FP32 → INT8
- **稀疏化**: 只传Top-K梯度
- **误差反馈**: 累积舍入误差

**权衡**: 通信 ↓, 精度 ↓ (通常可接受)

#### 4. 通信与计算重叠

**DDP自动优化**:
- 边计算边传输梯度
- Bucket机制分批传输

```python
model = DDP(model, 
           bucket_cap_mb=25,  # bucket大小
           gradient_as_bucket_view=True)  # 优化内存
```

### 5.2 调试技巧

#### 1. 验证数值一致性

```python
# 单GPU vs 多GPU loss应该相同(相同seed)
torch.manual_seed(42)
loss_single = train_single_gpu()

torch.manual_seed(42)
loss_multi = train_multi_gpu()

assert abs(loss_single - loss_multi) < 1e-5
```

#### 2. 监控GPU利用率

```bash
# 实时监控
watch -n 1 nvidia-smi

# 或使用Python
import pynvml
pynvml.nvmlInit()
handle = pynvml.nvmlDeviceGetHandleByIndex(0)
util = pynvml.nvmlDeviceGetUtilizationRates(handle)
print(f"GPU利用率: {util.gpu}%")
```

#### 3. Profiling分析

```python
from torch.profiler import profile, ProfilerActivity

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True
) as prof:
    model(input)

# 查看通信时间
print(prof.key_averages().table(
    sort_by="cuda_time_total", row_limit=10))
```

### 5.3 常见问题

**问题1**: GPU利用率低
- 检查DataLoader的num_workers
- 增大批大小
- 使用pin_memory
- Profiling找瓶颈

**问题2**: OOM (Out of Memory)
- 减小批大小
- 梯度累积
- 混合精度
- 梯度检查点

**问题3**: 通信瓶颈
- 减少同步频率(梯度累积)
- 梯度压缩
- 更快的互连(InfiniBand, NVLink)

**问题4**: 不同GPU速度不一致
- 使用同型号GPU
- 异步训练(如果收敛允许)
- 动态负载均衡

## 6. 小结

### 核心要点

1. **从零实现数据并行**
   - get_params: 参数分发
   - split_batch: 数据分割
   - allreduce: 梯度聚合
   - train_batch: 完整训练步骤

2. **PyTorch多GPU方案**
   - **nn.DataParallel**: 简单,单机,有GIL限制
   - **DistributedDataParallel**: 高效,多机,推荐
   - 自动数据分割和梯度聚合

3. **参数服务器**
   - Push-Pull模型
   - 多服务器分片
   - 键值存储抽象
   - 适合多机大规模训练

4. **性能优化**
   - 梯度累积(大批量)
   - 混合精度(快+省显存)
   - 梯度压缩(减通信)
   - 计算通信重叠(提速)

5. **调试与监控**
   - 验证数值一致性
   - 监控GPU利用率
   - Profiling分析瓶颈
   - 处理常见问题

### 实践建议

**入门**:
1. 从单GPU开始,确保代码正确
2. 使用nn.DataParallel快速扩展
3. 监控加速比和GPU利用率

**进阶**:
1. 迁移到DistributedDataParallel
2. 启用混合精度训练
3. 调优批大小和学习率

**专家**:
1. 自定义通信策略
2. 实现模型并行(大模型)
3. 探索3D并行(数据+模型+流水线)

### 推荐工具

- **Horovod**: 统一的分布式训练框架
- **DeepSpeed**: 微软的大模型训练优化
- **Megatron-LM**: NVIDIA的模型并行库
- **PyTorch Lightning**: 简化分布式训练代码
- **Ray**: 分布式计算框架

## 练习

1. **All-Reduce实现**: 实现Ring All-Reduce算法,对比性能。

2. **批大小实验**: 在1/2/4 GPU上,保持总批量512不变,对比收敛速度。

3. **学习率缩放**: 实验不同的学习率缩放策略(线性、平方根、不缩放)。

4. **混合精度**: 为LeNet添加混合精度训练,测量加速比。

5. **通信分析**: 使用Profiler分析多GPU训练的通信开销占比。

6. **DDP实践**: 将DataParallel改为DistributedDataParallel,编写启动脚本。

7. **梯度累积**: 实现梯度累积,用小批量模拟大批量训练。

8. **异步训练**: (挑战)实现简单的参数服务器和异步SGD。